# Klima-Komposita Analyse (final, refactored)

Ziele:
- klare Zellstruktur (Imports/Constants, Logik, Plotdaten, Plot)
- einmalige Berechnung, mehrfaches Plotten
- Fokus: 2023Q2 bis 2023Q4 (Klimaschutz vs Klimakrise)
- Plot: Zeitungen Veraenderung 2023-06 -> 2023-07 (Wandel vs Schutz) mit Labels fuer Zeitungen > 20
- Plot: erstes volles Quartal -> letztes volles Quartal + Pairwise Varianten

In [ ]:
import os
        {
            "cell_type": "code",
            "id": "#VSC-40b27c45",
            "metadata": {
                "language": "python"
            },
            "source": [
                "# Use helpers: compute deltas and plot with scatter_delta",
                "df_q2q3 = delta_between_periods(terms_df, '2023Q2', '2023Q3', 'quarter')",
                "scatter_delta(",
                "    df_q2q3,",
                "    'Klimakrise',",
                "    'Klimaschutz',",
                "    'Zeitungen: Veraenderung 2023Q2 -> 2023Q3 (Klimakrise vs Klimaschutz)',",
                "    label_threshold=20,",
                "    outname='delta_2023Q2_to_2023Q3_krise_vs_schutz.png',",
                ")"
            ]
        },
    x = dfc[f'delta_{term_x}']
    y = dfc[f'delta_{term_y}']
    fig, ax = plt.subplots(figsize=(12, 9))
    ax.axhline(0, color='lightgray', linewidth=0.8)
    ax.axvline(0, color='lightgray', linewidth=0.8)
    sizes = np.clip(np.sqrt(dfc['total_sum']), 1, 200) * 80
    ax.scatter(x, y, s=sizes, facecolors='#4D4D4D', edgecolors='black', linewidths=0.7, alpha=0.28)
    label_dset = dfc[dfc['total_sum'] > label_threshold]
    x_min, x_max = x.min(), x.max()
    y_min, y_max = y.min(), y.max()
    x_span = max(x_max - x_min, 1e-6)
    y_span = max(y_max - y_min, 1e-6)
    for _, r in label_dset.iterrows():
        ax.text(
            r[f'delta_{term_x}'],
            r[f'delta_{term_y}'],
            r['newspaper_name'],
            fontsize=8,
            ha='center',
            va='center',
            bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='none', alpha=0.9)
        )
    ax.set_xlabel(f'Delta {term_x} (pp)')
    ax.set_ylabel(f'Delta {term_y} (pp)')
    ax.set_title(title)
    ax.grid(True, alpha=0.25)
    handles = [plt.scatter([], [], s=s * 6, facecolors='#4D4D4D', edgecolors='black', alpha=0.28) for s in [3, 10, 30]]
    labels = ['~3 Nennungen (Sum)', '~10 Nennungen', '~30 Nennungen']
    ax.legend(handles, labels, loc='lower left', frameon=True, title='Punktgroesse')
    plt.tight_layout()
    if outname:
        outpath = os.path.join(plot_dir, outname)
        plt.savefig(outpath, dpi=300, bbox_inches='tight')
        print('Grafik gespeichert:', outpath)
    plt.show()

## Gesamtverteilung (Daten)

In [ ]:
counts_all = terms_df['begriff'].value_counts().reindex(TERMS, fill_value=0)
total_all = counts_all.sum()
percentages_all = (counts_all / total_all * 100).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bar_colors = ['#1A1A1A', '#7A7A7A', '#C9C9C9']
bars = ax.bar(counts_all.index, counts_all.values, color=bar_colors, edgecolor='white', linewidth=0.6)
for bar, pct in zip(bars, percentages_all.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(counts_all.max() * 0.02, 3),
        f'{pct}%',
        ha='center',
        va='bottom',
        fontsize=10,
        color='black',
    )
ax.set_title('Gesamtverteilung der Begriffe (Gesamtzeitraum)')
ax.set_ylabel('Absolute Haeufigkeit')
ax.set_xlabel('Begriff')
ax.set_ylim(0, counts_all.max() * 1.18)
ax.grid(axis='y', alpha=0.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
outpath = os.path.join(plot_dir, 'absolute_haeufigkeit_gesamt.png')
plt.savefig(outpath, dpi=300, bbox_inches='tight')
plt.show()
print('Grafik gespeichert:', outpath)

## Zeitliche Entwicklung (Gesamtzeitraum) - Monatswerte (Daten)

In [ ]:
monthly_counts_all = terms_df.groupby(['month', 'begriff']).size().unstack(fill_value=0)
for t in TERMS:
    if t not in monthly_counts_all.columns:
        monthly_counts_all[t] = 0
monthly_counts_all = monthly_counts_all.reindex(columns=TERMS, fill_value=0)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
x = range(len(monthly_counts_all))
labels = [str(m) for m in monthly_counts_all.index]
ax.plot(x, monthly_counts_all['Klimaschutz'], linestyle='-', marker='o', color='black', linewidth=1.6, markersize=3, label='Klimaschutz')
ax.plot(x, monthly_counts_all['Klimawandel'], linestyle='--', marker='s', color='dimgray', linewidth=1.6, markersize=3, label='Klimawandel')
ax.plot(x, monthly_counts_all['Klimakrise'], linestyle=':', marker='^', color='gray', linewidth=1.6, markersize=3, label='Klimakrise')
ax.set_xticks(x[::2])
ax.set_xticklabels(labels[::2], rotation=45, ha='right')
ax.set_xlabel('Monat')
ax.set_ylabel('Absolute Haeufigkeit')
ax.set_title('Monatliche absolute Haeufigkeiten (Gesamtzeitraum)')
ax.grid(True, alpha=0.3)
ax.legend(loc='best', frameon=True)
plt.tight_layout()
outpath = os.path.join(plot_dir, 'monatliche_haeufigkeiten_gesamt.png')
plt.savefig(outpath, dpi=300, bbox_inches='tight')
plt.show()
print('Grafik gespeichert:', outpath)

## Zeitliche Entwicklung (Gesamtzeitraum) - Quartalswerte (Daten)

In [ ]:
quarterly_counts_all = terms_df.groupby(['quarter', 'begriff']).size().unstack(fill_value=0)
for t in TERMS:
    if t not in quarterly_counts_all.columns:
        quarterly_counts_all[t] = 0
quarterly_counts_all = quarterly_counts_all.reindex(columns=TERMS, fill_value=0)
quarterly_total_all = quarterly_counts_all.sum(axis=1)
quarterly_pct_all = quarterly_counts_all.div(quarterly_total_all, axis=0).fillna(0) * 100

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
linestyles = {'Klimaschutz': '-', 'Klimawandel': '--', 'Klimakrise': ':'}
markers = {'Klimaschutz': 'o', 'Klimawandel': 's', 'Klimakrise': '^'}

for begriff in TERMS:
    ax.plot(
        range(len(quarterly_pct_all)),
        quarterly_pct_all[begriff],
        linestyle=linestyles[begriff],
        marker=markers[begriff],
        color='black',
        linewidth=1.5,
        markersize=5,
        label=begriff,
    )

ax.set_xticks(range(len(quarterly_pct_all)))
ax.set_xticklabels([str(q) for q in quarterly_pct_all.index], rotation=45, ha='right')
ax.set_xlabel('Quartal')
ax.set_ylabel('Relativer Anteil (%)')
ax.set_title('Entwicklung der Klima-Begriffe (Gesamtzeitraum)')
ax.legend(loc='best', frameon=True)
ax.grid(True, alpha=0.3)

plt.tight_layout()
outpath = os.path.join(plot_dir, 'quartalsanteile_gesamt.png')
plt.savefig(outpath, dpi=300, bbox_inches='tight')
plt.show()
print('Grafik gespeichert:', outpath)


## 2023Q2 bis 2023Q4: Klimaschutz vs Klimakrise (Daten)

In [ ]:
quarterly_counts = terms_df.groupby(['quarter', 'begriff']).size().unstack(fill_value=0)
quarterly_total = quarterly_counts.sum(axis=1)
quarterly_pct = quarterly_counts.div(quarterly_total, axis=0).fillna(0) * 100
q_focus = quarterly_pct.loc[quarterly_pct.index.isin(['2023Q2', '2023Q3', '2023Q4'])]
print('Anteile 2023Q2-2023Q4 (in %):')
print(q_focus[['Klimaschutz', 'Klimakrise']].round(2).to_string())

## 2023Q2 -> 2023Q3: Klimaschutz vs Klimakrise (Zeitungen, Shift)

In [ ]:
q2_counts = counts_for_period(terms_df, '2023Q2', 'quarter')
q3_counts = counts_for_period(terms_df, '2023Q3', 'quarter')
q2_sh = shares_from_counts(q2_counts)
q3_sh = shares_from_counts(q3_counts)
papers = sorted(set(q2_sh.index).union(q3_sh.index))
q2_sh = q2_sh.reindex(papers, fill_value=0)
q3_sh = q3_sh.reindex(papers, fill_value=0)
q2_counts = q2_counts.reindex(papers, fill_value=0)
q3_counts = q3_counts.reindex(papers, fill_value=0)

df_q2q3 = pd.DataFrame({'newspaper_name': papers})
df_q2q3['delta_Klimaschutz'] = q3_sh['Klimaschutz'].values - q2_sh['Klimaschutz'].values
df_q2q3['delta_Klimakrise'] = q3_sh['Klimakrise'].values - q2_sh['Klimakrise'].values
df_q2q3['total_krise_schutz'] = (
    q2_counts[['Klimaschutz', 'Klimakrise']].sum(axis=1).values
    + q3_counts[['Klimaschutz', 'Klimakrise']].sum(axis=1).values
 )

fig, ax = plt.subplots(figsize=(12, 9))
ax.axhline(0, color='lightgray', linewidth=0.8)
ax.axvline(0, color='lightgray', linewidth=0.8)
sizes = np.clip(np.sqrt(df_q2q3['total_krise_schutz']), 1, 200) * 6
ax.scatter(
    df_q2q3['delta_Klimaschutz'],
    df_q2q3['delta_Klimakrise'],
    s=sizes,
    facecolors='#4D4D4D',
    edgecolors='black',
    linewidths=0.7,
    alpha=0.28,
 )

label_threshold = 20
label_dset = df_q2q3[df_q2q3['total_krise_schutz'] > label_threshold]
x = df_q2q3['delta_Klimaschutz']
y = df_q2q3['delta_Klimakrise']
x_min, x_max = x.min(), x.max()
y_min, y_max = y.min(), y.max()
x_span = max(x_max - x_min, 1e-6)
y_span = max(y_max - y_min, 1e-6)
for _, r in label_dset.iterrows():
    dx = r['delta_Klimaschutz']
    dy = r['delta_Klimakrise']
    radius_pts = np.sqrt(max(1, r['total_krise_schutz'])) * 6
    offset_pts = max(12, 10 + 0.8 * radius_pts)
    x_sign = 1 if dx >= 0 else -1
    y_sign = 1 if dy >= 0 else -1
    if dx > x_min + 0.8 * x_span:
        x_sign = -1
    if dx < x_min + 0.2 * x_span:
        x_sign = 1
    if dy > y_min + 0.8 * y_span:
        y_sign = -1
    if dy < y_min + 0.2 * y_span:
        y_sign = 1
    ax.annotate(
        r['newspaper_name'],
        xy=(dx, dy),
        xytext=(x_sign * offset_pts, y_sign * offset_pts),
        textcoords='offset points',
        fontsize=8,
        ha='left' if x_sign > 0 else 'right',
        va='bottom' if y_sign > 0 else 'top',
        bbox=dict(boxstyle='round,pad=0.12', fc='white', ec='none', alpha=0.95),
        arrowprops=dict(arrowstyle='-', color='0.55', lw=0.6, shrinkA=0, shrinkB=0),
    )

ax.set_xlabel('Delta Klimaschutz (pp)')
ax.set_ylabel('Delta Klimakrise (pp)')
ax.set_title('Klimaschutz vs Klimakrise: Shift 2023Q3 - 2023Q2 (Zeitungen)')
ax.grid(True, alpha=0.25)
handles = [plt.scatter([], [], s=s * 6, facecolors='#4D4D4D', edgecolors='black', alpha=0.28) for s in [3, 10, 30]]
labels = ['~3 Nennungen (Q2+Q3)', '~10 Nennungen', '~30 Nennungen']
ax.legend(handles, labels, loc='lower left', frameon=True, title='Punktgroesse')
plt.tight_layout()
outpath = os.path.join(plot_dir, 'delta_2023Q2_to_2023Q3_schutz_vs_krise.png')
plt.savefig(outpath, dpi=300, bbox_inches='tight')
plt.show()
print('Grafik gespeichert:', outpath)

## 2023-06 -> 2023-07: Wandel vs Schutz (Daten)

In [ ]:
df_delta_jun_jul = delta_between_periods(terms_df, '2023-06', '2023-07', 'month')

In [ ]:
scatter_delta(
    df_delta_jun_jul,
    'Klimawandel',
    'Klimaschutz',
    'Zeitungen: Veraenderung 2023-06 -> 2023-07 (Wandel vs Schutz)',
    label_threshold=20,
    outname='delta_2023-06_to_2023-07_wandel_vs_schutz.png',
)

## Erstes volles Quartal -> letztes volles Quartal (Daten)

In [ ]:
min_date = terms_df['data_published'].min()
max_date = terms_df['data_published'].max()
all_quarters = pd.period_range(min_date.to_period('Q'), max_date.to_period('Q'), freq='Q')
full_quarters = [q for q in all_quarters if q.start_time >= min_date and q.end_time <= max_date]
if len(full_quarters) < 2:
    print('Nicht genug vollstaendige Quartale gefunden.')
    first_q = None
    last_q = None
    df_delta_q = None
else:
    first_q = str(full_quarters[0])
    last_q = str(full_quarters[-1])
    print('Vergleichsquartale:', first_q, '->', last_q)
    df_delta_q = delta_between_periods(terms_df, first_q, last_q, 'quarter')

## Pairwise Varianten (alle Kombinationen)

In [ ]:
if df_delta_q is not None:
    for term_x, term_y in combinations(TERMS, 2):
        scatter_delta(
            df_delta_q,
            term_x,
            term_y,
            f'Zeitungen: Veraenderung {first_q} -> {last_q} ({term_x} vs {term_y})',
            label_threshold=40,
            outname=f'delta_{first_q}_to_{last_q}_{term_x}_vs_{term_y}.png',
        )